In [54]:
OUTPUT_DIR = PROJECT_DIR / "Lab7_NumPy_Pandas_Additional_Exercises"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(OUTPUT_DIR)

d:\Downloads\OLERMO\Lab7_NumPy_Pandas_Olermo\Lab7_NumPy_Pandas_Additional_Exercises


In [55]:
import numpy as np
import pandas as pd

from pathlib import Path

In [56]:
from pathlib import Path

PROJECT_DIR = Path.cwd()

if not (PROJECT_DIR / "data" / "secom.data").exists():
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / "data" / "secom.data"
LABEL_PATH = PROJECT_DIR / "data" / "secom_labels.data"

print(DATA_PATH.exists())
print(LABEL_PATH.exists())

True
True


In [57]:
# Load manufacturing measurements
measurements = pd.read_csv(
    DATA_PATH,
    sep=r"\s+",
    header=None
)

# Load manufacturing labels
labels = pd.read_csv(
    LABEL_PATH,
    sep=r"\s+",
    header=None
)

# Extract original outcome labels
outcomes = labels.iloc[:, 0].to_numpy()

# Preserve original record identifiers
record_ids = np.arange(len(measurements))

# Construct labeled DataFrame
df = measurements.copy()

df.insert(0, "Record_ID", record_ids)
df["Outcome"] = outcomes

print("Measurement shape:", measurements.shape)
print("Label shape:", labels.shape)

print("\nOutcome distribution:")
print(pd.Series(outcomes).value_counts())

df.head()

Measurement shape: (1567, 590)
Label shape: (1567, 2)

Outcome distribution:
-1    1463
 1     104
Name: count, dtype: int64


,Record_ID,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Outcome
0,0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1


In [58]:
assert len(measurements) == len(labels)

print("Dataset and labels correctly aligned.")

Dataset and labels correctly aligned.


EXERCISE 1: VECTORIZED MEASUREMENT QUALITY ANALYSIS


In [59]:
# Identify features with missing values
missing_counts = measurements.isna().sum()

# Identify features containing constant values
unique_counts = measurements.nunique(dropna=True)

# Identify usable features
usable_features = measurements.columns[
    (unique_counts > 1) &
    (measurements.notna().sum() > 0)
].to_numpy()

print("Total measurement features:", measurements.shape[1])

print("Features containing missing values:",
      (missing_counts > 0).sum())

print("Constant features:",
      (unique_counts <= 1).sum())

print("Usable features:", len(usable_features))

Total measurement features: 590
Features containing missing values: 538
Constant features: 116
Usable features: 474


In [60]:
rng = np.random.default_rng(SEED_DIGIT)

number_of_features = 5 + (SEED_DIGIT % 3)

selected_features = rng.choice(
    usable_features,
    size=number_of_features,
    replace=False
)

selected_features = np.sort(selected_features)

print("Number of selected features:", len(selected_features))
print("Selected features:", selected_features)

Number of selected features: 7
Selected features: [ 11 253 288 353 376 467 470]


In [61]:
selected_data = measurements[selected_features]

# Calculate statistical measurements using NumPy
feature_means = np.nanmean(selected_data.to_numpy(), axis=0)

feature_medians = np.nanmedian(selected_data.to_numpy(), axis=0)

feature_stds = np.nanstd(
    selected_data.to_numpy(),
    axis=0,
    ddof=1
)

missing_percentages = (
    selected_data.isna().mean().to_numpy() * 100
)

print("Means:", feature_means)
print("Medians:", feature_medians)
print("Sample standard deviations:", feature_stds)
print("Missing percentages:", missing_percentages)

Means: [9.64352780e-01 3.32162022e-02 3.92365729e-03 4.03305250e-02
 1.60083067e-03 6.25209066e+00 5.36775176e+00]
Medians: [9.6580e-01 3.0800e-02 3.6000e-03 4.2100e-02 1.6000e-03 5.6450e+00
 4.0611e+00]
Sample standard deviations: [1.24520518e-02 2.24251057e-02 2.93609530e-03 1.54992731e-02
 5.34350073e-04 8.67372373e+00 4.98336669e+00]
Missing percentages: [0.12763242 1.53158902 0.19144863 1.53158902 0.12763242 0.25526484
 0.38289726]


In [62]:
# Convert selected measurements to NumPy
X = selected_data.to_numpy(dtype=float)

# Calculate median for each feature
medians = np.nanmedian(X, axis=0)

# Calculate absolute deviations
absolute_deviations = np.abs(X - medians)

# Calculate MAD
mad = np.nanmedian(absolute_deviations, axis=0)

# Prepare robust standardized values
robust_z = np.full(X.shape, np.nan)

# Identify features with nonzero MAD
nonzero_mad = mad > 0

# Calculate robust Z-scores
robust_z[:, nonzero_mad] = (
    0.6745 *
    (X[:, nonzero_mad] - medians[nonzero_mad])
    / mad[nonzero_mad]
)

# Handle zero MAD safely
zero_mad = ~nonzero_mad

robust_z[:, zero_mad] = np.where(
    np.isnan(X[:, zero_mad]),
    np.nan,
    np.where(
        X[:, zero_mad] == medians[zero_mad],
        0,
        np.inf
    )
)

print("Robust standardized values calculated.")
print("Robust Z-score shape:", robust_z.shape)

Robust standardized values calculated.
Robust Z-score shape: (1567, 7)


In [63]:
# Identify unusual observations
flagged = np.abs(robust_z) > 3.5

# Count flagged observations per feature
flagged_counts = np.sum(flagged, axis=0)

# Calculate flagged percentages
observed_counts = np.sum(~np.isnan(X), axis=0)

flagged_percentages = (
    flagged_counts / observed_counts
) * 100

print("Flagged observations:", flagged_counts)
print("Flagged percentages:", flagged_percentages)

Flagged observations: [ 5 26 13  3 10 55 52]
Flagged percentages: [0.31948882 1.68502916 0.83120205 0.19442644 0.63897764 3.51887396
 3.33119795]


In [64]:
# Construct the measurement quality summary
measurement_quality = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_Std": feature_stds,
    "MAD": mad,
    "Missing_Percentage": missing_percentages,
    "Flagged_Count": flagged_counts,
    "Flagged_Percentage": flagged_percentages
})

# Rank features by flagged percentage
measurement_quality["Rank"] = (
    measurement_quality["Flagged_Percentage"]
    .rank(ascending=False, method="min")
    .astype(int)
)

measurement_quality = measurement_quality.sort_values(
    "Rank"
)

measurement_quality

,Feature,Mean,Median,Sample_Std,MAD,Missing_Percentage,Flagged_Count,Flagged_Percentage,Rank
5,467,6.252091,5.6450,8.673724,0.8195,0.255265,55,3.518874,1
6,470,5.367752,4.0611,4.983367,1.5682,0.382897,52,3.331198,2
1,253,0.033216,0.0308,0.022425,0.0066,1.531589,26,1.685029,3
2,288,0.003924,0.0036,0.002936,0.0013,0.191449,13,0.831202,4
4,376,0.001601,0.0016,0.000534,0.0003,0.127632,10,0.638978,5
0,11,0.964353,0.9658,0.012452,0.0065,0.127632,5,0.319489,6
3,353,0.040331,0.0421,0.015499,0.0095,1.531589,3,0.194426,7


In [65]:
measurement_quality.to_csv(
    OUTPUT_DIR / "measurement_quality.csv",
    index=False
)

print("measurement_quality.csv successfully generated.")

measurement_quality.csv successfully generated.


In [66]:
total_flagged = int(flagged_counts.sum())

highest_missing = float(
    np.max(missing_percentages)
)

highest_flagged_feature = measurement_quality.iloc[0]["Feature"]

print("ASSESSMENT DATA")
print("-----------------------------")

print("Selected Features:", len(selected_features))

print("Highest Missing Percentage:",
      round(highest_missing, 2), "%")

print("Feature with Highest Flagged Percentage:",
      highest_flagged_feature)

print("Total Flagged Observations:",
      total_flagged)

print("Vectorized Results Verified:",
      np.allclose(
          feature_means,
          selected_data.mean().to_numpy(),
          equal_nan=True
      ))


ASSESSMENT DATA
-----------------------------
Selected Features: 7
Highest Missing Percentage: 1.53 %
Feature with Highest Flagged Percentage: 467.0
Total Flagged Observations: 164
Vectorized Results Verified: True


EXERCISE 2: MOBILE DEVICE USAGE PROFILING


In [67]:
# Use five selected features from Exercise 1
comparison_features = selected_features[:5]

print("Selected comparison features:")
print(comparison_features)

Selected comparison features:
[ 11 253 288 353 376]


In [68]:
print(df[["Record_ID", "Outcome"]].head())

print("\nOutcome counts:")
print(df["Outcome"].value_counts())

   Record_ID  Outcome
0          0       -1
1          1       -1
2          2        1
3          3       -1
4          4       -1

Outcome counts:
Outcome
-1    1463
 1     104
Name: count, dtype: int64


In [69]:
pass_indices = np.where(outcomes == 1)[0]

fail_indices = np.where(outcomes == -1)[0]

balanced_size = min(
    20 + ID_SUM,
    len(pass_indices),
    len(fail_indices)
)

rng2 = np.random.default_rng(SEED_DIGIT)

# Randomly select records from each outcome
selected_pass = rng2.choice(
    pass_indices,
    size=balanced_size,
    replace=False
)

selected_fail = rng2.choice(
    fail_indices,
    size=balanced_size,
    replace=False
)

# Combine selected original record indices
balanced_indices = np.concatenate([
    selected_pass,
    selected_fail
])

# Construct balanced sample
balanced_sample = df.iloc[balanced_indices][
    ["Record_ID", "Outcome"] + list(comparison_features)
].copy()

# Shuffle the resulting records reproducibly
balanced_sample = balanced_sample.sample(
    frac=1,
    random_state=SEED_DIGIT
).reset_index(drop=True)

print("Balanced sample size:", len(balanced_sample))

print("\nOutcome distribution:")
print(balanced_sample["Outcome"].value_counts())

Balanced sample size: 40

Outcome distribution:
Outcome
-1    20
 1    20
Name: count, dtype: int64


In [70]:
# Calculate medians from the full dataset
full_medians = measurements[comparison_features].median()

# Apply median imputation to the balanced sample
balanced_sample[comparison_features] = (
    balanced_sample[comparison_features]
    .fillna(full_medians)
)

print("Missing values after imputation:")
print(
    balanced_sample[comparison_features].isna().sum()
)

Missing values after imputation:
11     0
253    0
288    0
353    0
376    0
dtype: int64


In [71]:
balanced_sample.to_csv(
    OUTPUT_DIR / "balanced_sample.csv",
    index=False
)

print("balanced_sample.csv successfully generated.")

balanced_sample.csv successfully generated.


In [72]:
grouped_statistics = balanced_sample.groupby(
    "Outcome"
)[comparison_features].agg([
    "count",
    "mean",
    "median",
    "std"
])

grouped_statistics

11                                253                               \
        count      mean   median       std count      mean   median       std   
Outcome                                                                         
-1         20  0.964910  0.96710  0.009333    20  0.033355  0.03135  0.014746   
 1         20  0.962295  0.96435  0.010761    20  0.034680  0.03205  0.012414   

          288                               353                              \
        count      mean  median       std count      mean  median       std   
Outcome                                                                       
-1         20  0.004040  0.0035  0.001796    20  0.039725  0.0409  0.009412   
 1         20  0.004745  0.0044  0.003667    20  0.042380  0.0432  0.016902   

          376                              
        count     mean   median       std  
Outcome                                    
-1         20  0.00165  0.00165  0.000419  
 1         20  0.00170  0.00160  0.000538

In [73]:
pass_stats = balanced_sample[
    balanced_sample["Outcome"] == 1
][comparison_features].agg(
    ["count", "mean", "median", "std"]
)

fail_stats = balanced_sample[
    balanced_sample["Outcome"] == -1
][comparison_features].agg(
    ["count", "mean", "median", "std"]
)

absolute_mean_difference = (
    fail_stats.loc["mean"] -
    pass_stats.loc["mean"]
).abs()

In [74]:
n_pass = len(
    balanced_sample[balanced_sample["Outcome"] == 1]
)

n_fail = len(
    balanced_sample[balanced_sample["Outcome"] == -1]
)

pooled_std = np.sqrt(
    (
        (n_pass - 1) * pass_stats.loc["std"] ** 2
        +
        (n_fail - 1) * fail_stats.loc["std"] ** 2
    )
    /
    (n_pass + n_fail - 2)
)

# Handle zero denominators safely
standardized_difference = np.divide(
    absolute_mean_difference,
    pooled_std,
    out=np.where(
        absolute_mean_difference.to_numpy() == 0,
        0.0,
        np.inf
    ),
    where=pooled_std.to_numpy() > 0
)

print("Absolute mean differences:")
print(absolute_mean_difference)

print("\nStandardized mean differences:")
print(standardized_difference)

Absolute mean differences:
11     0.002615
253    0.001325
288    0.000705
353    0.002655
376    0.000050
Name: mean, dtype: float64

Standardized mean differences:
11     0.259629
253    0.097215
288    0.244184
353    0.194087
376    0.103725
dtype: float64


In [75]:
feature_comparison = pd.DataFrame({
    "Feature": comparison_features,
    "Pass_Count": pass_stats.loc["count"].values,
    "Fail_Count": fail_stats.loc["count"].values,
    "Pass_Mean": pass_stats.loc["mean"].values,
    "Fail_Mean": fail_stats.loc["mean"].values,
    "Pass_Median": pass_stats.loc["median"].values,
    "Fail_Median": fail_stats.loc["median"].values,
    "Pass_Std": pass_stats.loc["std"].values,
    "Fail_Std": fail_stats.loc["std"].values,
    "Absolute_Mean_Difference":
        absolute_mean_difference.values,
    "Pooled_Std": pooled_std.values,
    "Absolute_Standardized_Mean_Difference":
        standardized_difference
})

feature_comparison["Rank"] = (
    feature_comparison[
        "Absolute_Standardized_Mean_Difference"
    ].rank(
        ascending=False,
        method="min"
    ).astype(int)
)

feature_comparison = feature_comparison.sort_values(
    "Rank"
)

feature_comparison

,Feature,Pass_Count,Fail_Count,Pass_Mean,Fail_Mean,Pass_Median,Fail_Median,Pass_Std,Fail_Std,Absolute_Mean_Difference,Pooled_Std,Absolute_Standardized_Mean_Difference,Rank
11,11,20.0,20.0,0.962295,0.964910,0.96435,0.96710,0.010761,0.009333,0.002615,0.010072,0.259629,1
288,288,20.0,20.0,0.004745,0.004040,0.00440,0.00350,0.003667,0.001796,0.000705,0.002887,0.244184,2
353,353,20.0,20.0,0.042380,0.039725,0.04320,0.04090,0.016902,0.009412,0.002655,0.013679,0.194087,3
376,376,20.0,20.0,0.001700,0.001650,0.00160,0.00165,0.000538,0.000419,0.000050,0.000482,0.103725,4
253,253,20.0,20.0,0.034680,0.033355,0.03205,0.03135,0.012414,0.014746,0.001325,0.013630,0.097215,5


In [76]:
# Calculate means using NumPy
numpy_pass_means = np.mean(
    balanced_sample.loc[
        balanced_sample["Outcome"] == 1,
        comparison_features
    ].to_numpy(),
    axis=0
)

numpy_fail_means = np.mean(
    balanced_sample.loc[
        balanced_sample["Outcome"] == -1,
        comparison_features
    ].to_numpy(),
    axis=0
)

# Calculate means using Pandas
pandas_pass_means = pass_stats.loc["mean"].to_numpy()

pandas_fail_means = fail_stats.loc["mean"].to_numpy()

numpy_pandas_match = (
    np.allclose(numpy_pass_means, pandas_pass_means)
    and
    np.allclose(numpy_fail_means, pandas_fail_means)
)

print("NumPy-Pandas Statistics Match:",
      numpy_pandas_match)

NumPy-Pandas Statistics Match: True


In [77]:
feature_comparison.to_csv(
    OUTPUT_DIR / "feature_comparison.csv",
    index=False
)

print("feature_comparison.csv successfully generated.")

feature_comparison.csv successfully generated.


In [78]:
print("ASSESSMENT DATA")
print("-----------------------------")

print("Balanced Sample Size:", balanced_size)

print("Selected Features:", comparison_features)

print("Highest-Ranked Feature:",
      feature_comparison.iloc[0]["Feature"])

print("Absolute Standardized Mean Difference:",
      feature_comparison.iloc[0][
          "Absolute_Standardized_Mean_Difference"
      ])

print("NumPy-Pandas Statistics Match:",
      numpy_pandas_match)

ASSESSMENT DATA
-----------------------------
Balanced Sample Size: 20
Selected Features: [ 11 253 288 353 376]
Highest-Ranked Feature: 11.0
Absolute Standardized Mean Difference: 0.2596289584000915
NumPy-Pandas Statistics Match: True


EXERCISE 3: STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

In [79]:
# Identify features with at least 10 observed
# measurements in BOTH outcome groups

pass_data = measurements[outcomes == 1]
fail_data = measurements[outcomes == -1]

pass_counts = pass_data.notna().sum()
fail_counts = fail_data.notna().sum()

eligible_features = measurements.columns[
    (pass_counts >= 10) &
    (fail_counts >= 10) &
    (measurements.nunique(dropna=True) > 1)
].to_numpy()

# Select five reproducibly
rng3 = np.random.default_rng(SEED_DIGIT)

robustness_features = rng3.choice(
    eligible_features,
    size=5,
    replace=False
)

robustness_features = np.sort(robustness_features)

print("Selected features:")
print(robustness_features)

Selected features:
[ 11 254 383 469 472]


In [80]:
# Strategy 1: Available-case analysis
available_case_data = measurements[
    robustness_features
].copy()

# Strategy 2: Median imputation using the full dataset
imputed_data = measurements[
    robustness_features
].copy()

full_dataset_medians = measurements[
    robustness_features
].median()

imputed_data = imputed_data.fillna(
    full_dataset_medians
)

print("Available-case missing values:")
print(available_case_data.isna().sum())

print("\nImputed missing values:")
print(imputed_data.isna().sum())

Available-case missing values:
11        2
254       0
383    1018
469       6
472       7
dtype: int64

Imputed missing values:
11     0
254    0
383    0
469    0
472    0
dtype: int64


In [81]:
def calculate_mean_difference(data, feature):

    pass_values = data.loc[
        outcomes == 1, feature
    ].dropna().to_numpy()

    fail_values = data.loc[
        outcomes == -1, feature
    ].dropna().to_numpy()

    pass_mean = np.mean(pass_values)
    fail_mean = np.mean(fail_values)

    difference = fail_mean - pass_mean

    return pass_mean, fail_mean, difference

In [82]:
for feature in robustness_features:

    pass_mean, fail_mean, difference = (
        calculate_mean_difference(
            available_case_data,
            feature
        )
    )

    print("Feature:", feature)
    print("Pass Mean:", pass_mean)
    print("Fail Mean:", fail_mean)
    print("Mean Difference:", difference)
    print("---------------------------")

Feature: 11
Pass Mean: 0.9628307692307693
Fail Mean: 0.9644611225188228
Mean Difference: 0.0016303532880534988
---------------------------
Feature: 254
Pass Mean: 0.01320673076923077
Fail Mean: 0.013995557074504443
Mean Difference: 0.0007888263052736738
---------------------------
Feature: 383
Pass Mean: 0.4059933333333333
Fail Mean: 0.548846050096339
Mean Difference: 0.1428527167630057
---------------------------
Feature: 469
Pass Mean: 6.283320192307693
Fail Mean: 5.617964035689774
Mean Difference: -0.6653561566179187
---------------------------
Feature: 472
Pass Mean: 138.1059826923077
Fail Mean: 137.87286449175824
Mean Difference: -0.23311820054945542
---------------------------


In [83]:
BOOTSTRAP_REPETITIONS = 200 + (10 * SEED_DIGIT)

print(
    "Bootstrap repetitions:",
    BOOTSTRAP_REPETITIONS
)

Bootstrap repetitions: 250


In [84]:
def bootstrap_analysis(
    data,
    features,
    repetitions,
    seed
):

    rng = np.random.default_rng(seed)

    results = []

    for feature in features:

        # Separate the outcome groups
        pass_values = data.loc[
            outcomes == 1, feature
        ].dropna().to_numpy()

        fail_values = data.loc[
            outcomes == -1, feature
        ].dropna().to_numpy()

        if len(pass_values) < 10 or len(fail_values) < 10:
            raise ValueError(
                f"Insufficient observations for {feature}"
            )

        # Original mean difference
        observed_difference = (
            np.mean(fail_values) -
            np.mean(pass_values)
        )

        bootstrap_differences = np.empty(repetitions)

        # Bootstrap resampling
        for i in range(repetitions):

            pass_sample = rng.choice(
                pass_values,
                size=len(pass_values),
                replace=True
            )

            fail_sample = rng.choice(
                fail_values,
                size=len(fail_values),
                replace=True
            )

            bootstrap_differences[i] = (
                np.mean(fail_sample) -
                np.mean(pass_sample)
            )

        # Calculate percentile-based confidence interval
        lower_ci, upper_ci = np.percentile(
            bootstrap_differences,
            [2.5, 97.5]
        )

        results.append({
            "Feature": feature,
            "Mean_Difference": observed_difference,
            "CI_Lower": lower_ci,
            "CI_Upper": upper_ci,
            "Bootstrap_Repetitions": repetitions
        })

    return pd.DataFrame(results)

In [85]:
available_results = bootstrap_analysis(
    available_case_data,
    robustness_features,
    BOOTSTRAP_REPETITIONS,
    SEED_DIGIT
)

available_results["Strategy"] = "Available-case"

available_results

,Feature,Mean_Difference,CI_Lower,CI_Upper,Bootstrap_Repetitions,Strategy
0,11,0.001630,-0.000359,0.003482,250,Available-case
1,254,0.000789,-0.000812,0.002607,250,Available-case
2,383,0.142853,0.007168,0.313614,250,Available-case
3,469,-0.665356,-1.741682,0.017687,250,Available-case
4,472,-0.233118,-10.042970,9.576009,250,Available-case


In [86]:
imputed_results = bootstrap_analysis(
    imputed_data,
    robustness_features,
    BOOTSTRAP_REPETITIONS,
    SEED_DIGIT
)

imputed_results["Strategy"] = "Median imputation"

imputed_results

,Feature,Mean_Difference,CI_Lower,CI_Upper,Bootstrap_Repetitions,Strategy
0,11,0.001632,-0.000279,0.003321,250,Median imputation
1,254,0.000789,-0.000931,0.002491,250,Median imputation
2,383,0.052891,0.001534,0.110364,250,Median imputation
3,469,-0.665953,-1.734952,0.086931,250,Median imputation
4,472,-0.231289,-9.199627,10.831992,250,Median imputation


In [87]:
# Combine both strategies
robustness_comparison = pd.concat(
    [
        available_results,
        imputed_results
    ],
    ignore_index=True
)

# Rank features within each strategy
robustness_comparison["Rank"] = (
    robustness_comparison.groupby("Strategy")[
        "Mean_Difference"
    ].transform(
        lambda x: x.abs().rank(
            ascending=False,
            method="min"
        )
    )
)

robustness_comparison["Rank"] = (
    robustness_comparison["Rank"].astype(int)
)

robustness_comparison

,Feature,Mean_Difference,CI_Lower,CI_Upper,Bootstrap_Repetitions,Strategy,Rank
0,11,0.001630,-0.000359,0.003482,250,Available-case,4
1,254,0.000789,-0.000812,0.002607,250,Available-case,5
2,383,0.142853,0.007168,0.313614,250,Available-case,3
3,469,-0.665356,-1.741682,0.017687,250,Available-case,1
4,472,-0.233118,-10.042970,9.576009,250,Available-case,2
5,11,0.001632,-0.000279,0.003321,250,Median imputation,4
6,254,0.000789,-0.000931,0.002491,250,Median imputation,5
7,383,0.052891,0.001534,0.110364,250,Median imputation,3
8,469,-0.665953,-1.734952,0.086931,250,Median imputation,1
9,472,-0.231289,-9.199627,10.831992,250,Median imputation,2


In [88]:
# Compare available-case and imputed results
comparison_pivot = robustness_comparison.pivot(
    index="Feature",
    columns="Strategy",
    values="Mean_Difference"
)

# Identify changes in direction
changed_sign = (
    np.sign(
        comparison_pivot["Available-case"]
    ) !=
    np.sign(
        comparison_pivot["Median imputation"]
    )
)

# Calculate changes in mean differences
mean_difference_change = (
    comparison_pivot["Available-case"] -
    comparison_pivot["Median imputation"]
).abs()

# Compare rankings
rank_pivot = robustness_comparison.pivot(
    index="Feature",
    columns="Strategy",
    values="Rank"
)

changed_ranking = (
    rank_pivot["Available-case"] !=
    rank_pivot["Median imputation"]
)

print("Features with changed difference signs:")
print(comparison_pivot.index[changed_sign].tolist())

print("\nFeatures with changed rankings:")
print(rank_pivot.index[changed_ranking].tolist())

print("\nLargest change in mean difference:")
print(mean_difference_change.max())

Features with changed difference signs:
[]

Features with changed rankings:
[]

Largest change in mean difference:
0.08996203244732143


In [89]:
verification_results = bootstrap_analysis(
    available_case_data,
    robustness_features,
    BOOTSTRAP_REPETITIONS,
    SEED_DIGIT
)

reproducibility_verified = np.allclose(
    available_results[
        ["Mean_Difference", "CI_Lower", "CI_Upper"]
    ].to_numpy(),
    verification_results[
        ["Mean_Difference", "CI_Lower", "CI_Upper"]
    ].to_numpy()
)

print("Reproducibility Verified:",
      reproducibility_verified)

Reproducibility Verified: True


In [90]:
robustness_comparison.to_csv(
    OUTPUT_DIR / "robustness_comparison.csv",
    index=False
)

print("robustness_comparison.csv successfully generated.")

robustness_comparison.csv successfully generated.


In [91]:
largest_change = mean_difference_change.max()

changed_sign_features = (
    comparison_pivot.index[changed_sign].tolist()
)

changed_rank_features = (
    rank_pivot.index[changed_ranking].tolist()
)

report = f"""
STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS
================================================

Selected Features:
{robustness_features.tolist()}

Bootstrap Repetitions:
{BOOTSTRAP_REPETITIONS}

Preparation Strategies:
1. Available-case analysis
2. Median imputation using full dataset medians

Largest Change in Mean Difference:
{largest_change:.6f}

Features with Changed Difference Sign:
{changed_sign_features}

Features with Changed Ranking:
{changed_rank_features}

Reproducibility Verified:
{reproducibility_verified}

INTERPRETATION:
The analysis compared manufacturing outcome differences
under available-case analysis and median imputation.

Differences in mean estimates and feature rankings indicate
the sensitivity of the results to data preparation.

The bootstrap confidence intervals describe the uncertainty
associated with the estimated mean differences.

The same random seed was used to verify reproducibility.
"""

with open(
    OUTPUT_DIR / "sensitivity_report.txt",
    "w",
    encoding="utf-8"
) as file:

    file.write(report)

print(report)
print("sensitivity_report.txt successfully generated.")


STATISTICAL ROBUSTNESS AND SENSITIVITY ANALYSIS

Selected Features:
[11, 254, 383, 469, 472]

Bootstrap Repetitions:
250

Preparation Strategies:
1. Available-case analysis
2. Median imputation using full dataset medians

Largest Change in Mean Difference:
0.089962

Features with Changed Difference Sign:
[]

Features with Changed Ranking:
[]

Reproducibility Verified:
True

INTERPRETATION:
The analysis compared manufacturing outcome differences
under available-case analysis and median imputation.

Differences in mean estimates and feature rankings indicate
the sensitivity of the results to data preparation.

The bootstrap confidence intervals describe the uncertainty
associated with the estimated mean differences.

The same random seed was used to verify reproducibility.

sensitivity_report.txt successfully generated.


In [92]:
required_files = [
    "measurement_quality.csv",
    "balanced_sample.csv",
    "feature_comparison.csv",
    "robustness_comparison.csv",
    "sensitivity_report.txt"
]

print("FINAL OUTPUT VERIFICATION")
print("==========================")

for filename in required_files:

    file_path = OUTPUT_DIR / filename

    print(
        filename,
        "->",
        "EXISTS" if file_path.exists() else "MISSING"
    )

FINAL OUTPUT VERIFICATION
measurement_quality.csv -> EXISTS
balanced_sample.csv -> EXISTS
feature_comparison.csv -> EXISTS
robustness_comparison.csv -> EXISTS
sensitivity_report.txt -> EXISTS


In [93]:
print("EXERCISE VERIFICATION")
print("=====================")

print("Exercise 1:")
print("Selected features:", len(selected_features))

print("\nExercise 2:")
print("Balanced sample:", len(balanced_sample))
print("Equal group sizes:",
      n_pass == n_fail)
print("NumPy-Pandas match:",
      numpy_pandas_match)

print("\nExercise 3:")
print("Selected features:", len(robustness_features))
print("Bootstrap repetitions:",
      BOOTSTRAP_REPETITIONS)
print("Reproducibility:",
      reproducibility_verified)

EXERCISE VERIFICATION
Exercise 1:
Selected features: 7

Exercise 2:
Balanced sample: 40
Equal group sizes: True
NumPy-Pandas match: True

Exercise 3:
Selected features: 5
Bootstrap repetitions: 250
Reproducibility: True
